In [33]:
import pandas as pd
import pickle

X_test  = pd.read_parquet("X_test.parquet")
y_test  = pd.read_parquet("y_test.parquet")

# dans l'approche LLM, il n'est pas possible de prédire un Tag sans le texte de réclamation
filter = (X_test["Consumer Claim"].isna() == False)
X_test  = X_test[filter]
y_test  = y_test[filter]

# Rechargement
with open("categories.pkl", "rb") as f:
    categories = pickle.load(f)

# Approche LLM

On demande au LLM de choisir parmis la catégorie 'Tag' en fonction de la demande client 'Consumer Claim'

In [34]:
import os
import pandas as pd

from mistralai.client import Mistral


# Client Mistral
client = Mistral(
    api_key=os.environ["MISTRAL_API_KEY"]
)


# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

SYSTEM_PROMPT = """
Vous êtes un système de classification de réclamations clientes.

Votre tâche consiste à attribuer à chaque réclamation UNE SEULE catégorie parmi les catégories autorisées.

Vous devez retourner exactement le nom d'une catégorie présente dans la liste fournie, sans explication supplémentaire.

Catégories autorisées :
{categories}
"""


# ---------------------------------------------------------
# Fonction de classification
# ---------------------------------------------------------

def classify_with_llm(
    claim: str,
    categories: list[str],
    model: str = "mistral-small-latest",
    temperature: float = 0.0,
) -> str:

    system_prompt = SYSTEM_PROMPT.format(
        categories="\n".join(f"- {category}" for category in categories)
    )

    user_prompt = f"""
Réclamation à classifier :

{claim}

Retournez uniquement la catégorie correspondante.
"""

    response = client.chat.complete(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    return response.choices[0].message.content.strip()

In [35]:
X_test.head(10)

,Consumer Claim,Company,Date received,Submitted via,Tags,State
192075,I generally let people walk over me you could ...,JPMORGAN CHASE & CO.,2018-07-23,Web,NaN,FL
131727,MR. XXXX calls and tells me he is with the leg...,Critical Resolution Mediation LLC,2018-10-19,Web,NaN,TX
455266,I am including my marriage license per your re...,"EQUIFAX, INC.",2017-07-25,Web,NaN,TN
80575,Disputed with company on XX/XX/XXXX. The compa...,WELLS FARGO & COMPANY,2019-01-09,Web,NaN,CA
551360,"On XXXX XXXX, XXXX, we turned-over our XXXX XX...","SUNTRUST BANKS, INC.",2017-02-22,Web,Older American,GA
694534,Transunion deleted XXXX XXXX and XXXX. I have ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",2016-06-15,Web,Older American,FL
776888,I contacted CFPB two years ago about how Bayvi...,"BAYVIEW LOAN SERVICING, LLC",2016-01-04,Web,NaN,GA
192053,We have received multiple calls from Commerica...,Commercial Acceptance Company,2018-07-23,Web,NaN,PA
721113,XXXX alleged that I owe them {$76.00}. for a p...,"Southwest Credit Systems, L.P.",2016-04-24,Web,Older American,NY
49478,To whom it may concern On XX/XX/XXXX Radius Gl...,Radius Global Solutions LLC,2019-02-26,Web,NaN,FL


# Test avec 1 ligne de données du dataset

In [ ]:
question = X_test["Consumer Claim"].iloc[10]
response = classify_with_llm(question, categories)
expected = y_test["Tag"].iloc[10]

print("Question :", question)
print("Réponse  :", response)
print("Attendue :", expected)

Question : I have repeatedly had this debt removed, because I dont owe this company anything. They continue to place this item back on my credit and take it off each time I do this. Making this the fourth time its being removed. I received documentation stating that they have no record of me owing anything. I know for a fact I dont owe the debt. This continues to be an issue with this company.
Réponse  : Credit reporting, credit repair services, or other personal consumer reports
Attendue : Debt collection


# Test avec 20 lignes de données du dataset

In [ ]:
indices = X_test.index[10:20]

good = 0

i = 1
for idx in indices :
    X = X_test.loc[idx]
    y = y_test.loc[idx]

    question = X["Consumer Claim"]
    response = classify_with_llm(question, categories)
    expected = y["Tag"]

    if response == expected:
        good = good + 1

    print("----------------------------- ")
    print("Essai ", i)
    print("----------------------------- ")
    print("Question :", question)
    print("Réponse  :", response)
    print("Attendue :", expected)
    print("\n")

    i=i+1

print("\n\n----------------------------- ")
print("Bonnes réponses ", good, "/", indices.shape[0])

----------------------------- 
Essai  0
----------------------------- 
Question : I have repeatedly had this debt removed, because I dont owe this company anything. They continue to place this item back on my credit and take it off each time I do this. Making this the fourth time its being removed. I received documentation stating that they have no record of me owing anything. I know for a fact I dont owe the debt. This continues to be an issue with this company.
Attendue : Debt collection
Réponse  : Debt collection


----------------------------- 
Essai  1
----------------------------- 
Question : It was brought to my attention, a local retailer in XXXX NJ, XXXX XXXX, was allowing my ex to use my XXXX XXXX card issued by Citibank. As soon as I was made aware of this, I contacted Citibank as of XX/XX/XXXX as I did not authorize these purchases and or was/is my signature on file! I do not and NEVER have and an authorized user on my card either. I was made aware of this when my ex got ar